# Prapemrosesan: *Cloud Masking* SCL + *Gap Filling* PCHIP

Alur notebook:
1. **Konfigurasi + verifikasi anchor** — hitung % awan per tanggal (SCL 3/8/9/10 + *buffer* 30 m), tanggal dengan awan ≤ `ANCHOR_CLOUD_MAX` menjadi *anchor*.
2. **Fungsi PCHIP manual** — interpolasi Hermite kubik monoton (setara `scipy.interpolate.PchipInterpolator`).
3. **Verifikasi PCHIP** — bandingkan dengan SciPy.
4. **Persiapan raster + pemrosesan *blockwise*** — isi piksel berawan setiap tanggal memakai nilai *anchor* asli.
5. **Main process** — jalankan untuk 3 wilayah × 9 band.

Kode metode pengisian (`method`): `0` = tidak terisi, `1` = PCHIP, `2` = *backward fill*, `3` = *forward fill*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### **1. Konfigurasi + verifikasi anchor**

In [ ]:
import os, re, glob, shutil, time, gc
from datetime import datetime
from contextlib import ExitStack

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from scipy.ndimage import distance_transform_edt

# ============================================================
# Konfigurasi path & parameter
# ============================================================
SKRIPSI_DIR = "/content/drive/MyDrive/Colab Notebooks/skripsi"
REGIONS = ["kota_tangerang", "kota_tangerang_selatan", "kabupaten_tangerang"]
REGION_DIRS = {r: os.path.join(SKRIPSI_DIR, r) for r in REGIONS}
PCHIP_OUT_DIRS = {r: os.path.join(SKRIPSI_DIR, f"{r}_pchip") for r in REGIONS}

BANDS_TO_PROCESS = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B11", "B12"]

# SCL: 3 = cloud shadow, 8 = cloud medium prob., 9 = cloud high prob., 10 = thin cirrus
CLOUD_RELATED_SCL = {3, 8, 9, 10}
CLOUD_CLASSES_ARRAY = np.array(sorted(CLOUD_RELATED_SCL), dtype=np.uint8)

CLOUD_BUFFER_METERS = 30.0  # hasil audit residual cloud edge
ANCHOR_CLOUD_MAX = 6.0      # % awan maks. SETELAH buffer agar tanggal menjadi anchor
BLOCK_SIZE = 256            # ukuran window blockwise (aman untuk Colab Free)
PROCESS_VERSION = "pchip_manual_cloudbuffer30m_v2"  # agar output lama tidak dianggap selesai
FORCE_REPROCESS = False     # True = paksa proses ulang

# Anchor manual (opsional)
USE_MANUAL_ANCHORS = False
MANUAL_ANCHORS_PER_REGION = {r: [] for r in REGIONS}

# ============================================================
# Helper umum
# ============================================================
def print_header(title, char="="):
    print(f"\n{char * 70}\n{title}\n{char * 70}")


def date_to_ordinal(date_str):
    """'YYYY-MM-DD' -> nomor hari (float), dipakai sebagai sumbu waktu PCHIP."""
    return float(datetime.strptime(date_str, "%Y-%m-%d").toordinal())


def iter_windows(width, height, block_size=BLOCK_SIZE):
    for row_off in range(0, height, block_size):
        for col_off in range(0, width, block_size):
            yield Window(col_off=col_off, row_off=row_off,
                         width=min(block_size, width - col_off),
                         height=min(block_size, height - row_off))


def get_region_dates(region_dir):
    """Daftar folder tanggal berformat YYYY-MM-DD, terurut."""
    return sorted(
        name for name in os.listdir(region_dir)
        if os.path.isdir(os.path.join(region_dir, name))
        and re.fullmatch(r"\d{4}-\d{2}-\d{2}", name)
    )


def find_one_file(pattern, label):
    files = glob.glob(pattern)
    if len(files) == 0:
        raise FileNotFoundError(f"Tidak ditemukan file {label}:\n{pattern}")
    if len(files) > 1:
        raise RuntimeError(f"Ditemukan lebih dari 1 file untuk {label}:\n{files}")
    return files[0]


def expand_window(window, width, height, halo):
    """Perluas window sebanyak `halo` piksel (dibatasi tepi raster).
    Mengembalikan window luas + slice untuk memotong kembali ke window inti."""
    row_off, col_off = int(window.row_off), int(window.col_off)
    win_h, win_w = int(window.height), int(window.width)
    row_start, col_start = max(0, row_off - halo), max(0, col_off - halo)
    row_end = min(height, row_off + win_h + halo)
    col_end = min(width, col_off + win_w + halo)
    expanded = Window(col_off=col_start, row_off=row_start,
                      width=col_end - col_start, height=row_end - row_start)
    crop_rows = slice(row_off - row_start, row_off - row_start + win_h)
    crop_cols = slice(col_off - col_start, col_off - col_start + win_w)
    return expanded, crop_rows, crop_cols

# ============================================================
# Cloud mask final: SCL 3/8/9/10 + buffer Euclidean 30 m
# ============================================================
def read_final_cloud_mask_window(scl_src, window):
    """Return (final_cloud, valid, base_cloud, scl) untuk satu window.
    Window dibaca dengan halo agar buffer di tepi blok tidak terpotong."""
    res_x, res_y = abs(float(scl_src.transform.a)), abs(float(scl_src.transform.e))
    halo = int(np.ceil(CLOUD_BUFFER_METERS / min(res_x, res_y)))
    expanded, crop_rows, crop_cols = expand_window(window, scl_src.width, scl_src.height, halo)

    scl_ext = scl_src.read(1, window=expanded)
    valid_ext = scl_ext != 0  # SCL 0 = NoData (di luar wilayah)
    base_cloud_ext = np.isin(scl_ext, CLOUD_CLASSES_ARRAY) & valid_ext

    if np.any(base_cloud_ext):
        # jarak (meter) setiap piksel ke piksel awan terdekat
        distance_m = distance_transform_edt(~base_cloud_ext, sampling=(res_y, res_x))
        final_cloud_ext = (distance_m <= CLOUD_BUFFER_METERS) & valid_ext
    else:
        final_cloud_ext = np.zeros_like(base_cloud_ext, dtype=bool)

    core = (crop_rows, crop_cols)
    return final_cloud_ext[core], valid_ext[core], base_cloud_ext[core], scl_ext[core]


def compute_region_cloud_stats(region_dir, date_name):
    """Persentase awan (SCL saja dan SCL + buffer) terhadap piksel valid wilayah."""
    scl_path = find_one_file(os.path.join(region_dir, date_name, "*_SCL_10m.tif"), f"SCL {date_name}")
    total_valid = total_base_cloud = total_final_cloud = 0
    with rasterio.open(scl_path) as src:
        for window in iter_windows(src.width, src.height):
            final_cloud, valid, base_cloud, _ = read_final_cloud_mask_window(src, window)
            total_valid += int(valid.sum())
            total_base_cloud += int(base_cloud.sum())
            total_final_cloud += int(final_cloud.sum())
    if total_valid == 0:
        return None
    return {
        "valid_pixels": total_valid,
        "base_cloud_pixels": total_base_cloud,
        "final_cloud_pixels": total_final_cloud,
        "base_cloud_pct": total_base_cloud / total_valid * 100,
        "final_cloud_pct": total_final_cloud / total_valid * 100,
    }

# ============================================================
# Deteksi seluruh tanggal
# ============================================================
region_dates = {}
for region_name, region_dir in REGION_DIRS.items():
    dates = get_region_dates(region_dir)
    if len(dates) == 0:
        raise RuntimeError(f"Tidak ditemukan folder tanggal pada {region_dir}")
    region_dates[region_name] = dates

ALL_DATES = sorted(set().union(*region_dates.values()))
print(f"Total tanggal unik ditemukan: {len(ALL_DATES)}")
print(f"Tanggal pertama : {ALL_DATES[0]}")
print(f"Tanggal terakhir: {ALL_DATES[-1]}")
print(f"Cloud buffer    : {CLOUD_BUFFER_METERS:.0f} meter")
print(f"Block size      : {BLOCK_SIZE} x {BLOCK_SIZE}")

# ============================================================
# Hitung % awan + tentukan status anchor
# ============================================================
results = []
for region_name, region_dir in REGION_DIRS.items():
    print_header(f"VERIFIKASI CLOUD + BUFFER 30 m: {region_name}")
    available_dates = set(region_dates[region_name])
    for date_name in ALL_DATES:
        stats = compute_region_cloud_stats(region_dir, date_name) if date_name in available_dates else None
        if stats is None:
            row = {"region": region_name, "tanggal": date_name, "cloud_pct_scl": None,
                   "cloud_pct_buffer30m": None, "tambahan_buffer_pct": None, "status": "TIDAK ADA DATA"}
            print(f"{date_name} | TIDAK ADA DATA")
        else:
            final_pct = stats["final_cloud_pct"]
            row = {
                "region": region_name,
                "tanggal": date_name,
                "cloud_pct_scl": round(stats["base_cloud_pct"], 2),
                "cloud_pct_buffer30m": round(final_pct, 2),
                "tambahan_buffer_pct": round(final_pct - stats["base_cloud_pct"], 2),
                "status": "ANCHOR (bersih)" if final_pct <= ANCHOR_CLOUD_MAX else "perlu gap-fill",
            }
            print(f"{date_name} | SCL={row['cloud_pct_scl']:.2f}% | "
                  f"+buffer={row['cloud_pct_buffer30m']:.2f}% | {row['status']}")
        results.append(row)

df_check = pd.DataFrame(results)
anchor_csv_path = os.path.join(SKRIPSI_DIR, "verifikasi_anchor_dates_buffer30m.csv")
df_check.to_csv(anchor_csv_path, index=False)
print(f"\nHasil verifikasi anchor baru disimpan:\n{anchor_csv_path}")

# ============================================================
# Daftar anchor final per wilayah
# ============================================================
anchor_dates_per_region = {}
for region_name in REGION_DIRS:
    if USE_MANUAL_ANCHORS:
        anchors = sorted(MANUAL_ANCHORS_PER_REGION[region_name])
        if len(anchors) == 0:
            raise ValueError(f"Manual anchor {region_name} masih kosong.")
        missing = [d for d in anchors if d not in region_dates[region_name]]
        if missing:
            raise ValueError(f"Anchor tidak ditemukan pada {region_name}: {missing}")
    else:
        is_anchor = (df_check["region"] == region_name) & (df_check["status"] == "ANCHOR (bersih)")
        anchors = sorted(df_check.loc[is_anchor, "tanggal"].tolist())

    if len(anchors) < 2:
        raise ValueError(f"{region_name} hanya memiliki {len(anchors)} anchor setelah buffer 30 m. "
                         f"Periksa threshold anchor.")
    anchor_dates_per_region[region_name] = anchors
    print_header(f"ANCHOR FINAL: {region_name}")
    print(f"Total anchor: {len(anchors)}")
    print(anchors)

### **2. Fungsi Piecewise Cubic Hermite Interpolating Polynomial (PCHIP)**

In [ ]:
# ============================================================
# PCHIP manual (Fritsch–Carlson), setara scipy.interpolate.PchipInterpolator
# ============================================================
def pchip_internal_slope(h_prev, h_next, d_prev, d_next):
    """Turunan di titik interior: rata-rata harmonik berbobot dari dua secant.
    Bernilai 0 jika secant berlawanan tanda atau salah satunya nol (menjaga monotonisitas)."""
    slope = np.zeros_like(d_prev, dtype=np.float64)
    same_direction = (d_prev != 0) & (d_next != 0) & (np.sign(d_prev) == np.sign(d_next))
    if np.any(same_direction):
        hp, hn = h_prev[same_direction], h_next[same_direction]
        dp, dn = d_prev[same_direction], d_next[same_direction]
        w1, w2 = 2.0 * hn + hp, hn + 2.0 * hp
        slope[same_direction] = (w1 + w2) / (w1 / dp + w2 / dn)
    return slope


def pchip_edge_slope(h0, h1, d0, d1):
    """Turunan di titik ujung: formula tiga titik satu sisi + pembatas monotonisitas."""
    slope = ((2.0 * h0 + h1) * d0 - h0 * d1) / (h0 + h1)
    wrong_direction = np.sign(slope) != np.sign(d0)
    limited = (~wrong_direction) & (np.sign(d0) != np.sign(d1)) & (np.abs(slope) > 3.0 * np.abs(d0))
    slope[wrong_direction] = 0.0
    slope[limited] = 3.0 * d0[limited]
    return slope


def pchip_manual_from_anchors(anchor_times, anchor_values, anchor_valid, target_time):
    """Isi nilai pada `target_time` untuk banyak piksel sekaligus (vektorisasi).

    anchor_times  : (n_anchor,)          waktu anchor (ordinal hari)
    anchor_values : (n_anchor, n_pixel)  nilai reflektansi anchor
    anchor_valid  : (n_anchor, n_pixel)  True jika anchor valid (tidak berawan) pada piksel tsb.
    Return        : output (n_pixel,), method (n_pixel,)
                    method: 0 = tidak terisi, 1 = PCHIP, 2 = backward fill, 3 = forward fill
    """
    x = np.asarray(anchor_times, dtype=np.float64)
    y = np.asarray(anchor_values, dtype=np.float64)
    valid = np.asarray(anchor_valid, dtype=bool)
    n_anchor, n_pixel = y.shape
    output = np.full(n_pixel, np.nan, dtype=np.float64)
    method = np.zeros(n_pixel, dtype=np.uint8)
    anchor_index = np.arange(n_anchor, dtype=np.int32)[:, None]

    # --- 2 anchor valid terdekat sebelum (j0 < j1) dan sesudah (j2 < j3) target, per piksel ---
    before_mask = valid & (x[:, None] < target_time)
    after_mask = valid & (x[:, None] > target_time)
    j1 = np.where(before_mask, anchor_index, -1).max(axis=0)
    j2 = np.where(after_mask, anchor_index, n_anchor).min(axis=0)
    j0 = np.where(before_mask & (anchor_index < j1[None, :]), anchor_index, -1).max(axis=0)
    j3 = np.where(after_mask & (anchor_index > j2[None, :]), anchor_index, n_anchor).min(axis=0)
    has_before, has_after = j1 >= 0, j2 < n_anchor

    # --- Hanya ada anchor sesudah -> backward fill (nilai anchor sesudah terdekat) ---
    only_after = (~has_before) & has_after
    if np.any(only_after):
        cols = np.where(only_after)[0]
        output[cols] = y[j2[cols], cols]
        method[cols] = 2

    # --- Hanya ada anchor sebelum -> forward fill (nilai anchor sebelum terdekat) ---
    only_before = has_before & (~has_after)
    if np.any(only_before):
        cols = np.where(only_before)[0]
        output[cols] = y[j1[cols], cols]
        method[cols] = 3

    # --- Ada anchor sebelum & sesudah -> PCHIP ---
    cols = np.where(has_before & has_after)[0]
    if len(cols) == 0:
        return output, method

    k0, k1, k2, k3 = j0[cols], j1[cols], j2[cols], j3[cols]
    x1, x2 = x[k1], x[k2]
    y1, y2 = y[k1, cols], y[k2, cols]
    h = x2 - x1
    d = (y2 - y1) / h
    m1, m2 = d.copy(), d.copy()  # default jika hanya 2 titik = linear
    has0, has3 = k0 >= 0, k3 < n_anchor

    def secant(a, b, c):
        """Jarak waktu dan kemiringan antara anchor a -> b pada kolom piksel c."""
        hab = x[b] - x[a]
        return hab, (y[b, c] - y[a, c]) / hab

    # Turunan m1 di anchor sebelum (x1)
    use = has0  # interior: x0, x1, x2
    if np.any(use):
        c = cols[use]
        h_prev, d_prev = secant(k0[use], k1[use], c)
        h_next, d_next = secant(k1[use], k2[use], c)
        m1[use] = pchip_internal_slope(h_prev, h_next, d_prev, d_next)
    use = (~has0) & has3  # endpoint kiri: x1, x2, x3
    if np.any(use):
        c = cols[use]
        h0, d0 = secant(k1[use], k2[use], c)
        h1, d1 = secant(k2[use], k3[use], c)
        m1[use] = pchip_edge_slope(h0, h1, d0, d1)

    # Turunan m2 di anchor sesudah (x2)
    use = has3  # interior: x1, x2, x3
    if np.any(use):
        c = cols[use]
        h_prev, d_prev = secant(k1[use], k2[use], c)
        h_next, d_next = secant(k2[use], k3[use], c)
        m2[use] = pchip_internal_slope(h_prev, h_next, d_prev, d_next)
    use = (~has3) & has0  # endpoint kanan: x2, x1, x0 (arah terbalik)
    if np.any(use):
        c = cols[use]
        h0, d0 = secant(k1[use], k2[use], c)
        h1, d1 = secant(k0[use], k1[use], c)
        m2[use] = pchip_edge_slope(h0, h1, d0, d1)

    # --- Basis cubic Hermite pada interval [x1, x2] ---
    s = (target_time - x1) / h
    h00 = 2 * s**3 - 3 * s**2 + 1
    h10 = s**3 - 2 * s**2 + s
    h01 = -2 * s**3 + 3 * s**2
    h11 = s**3 - s**2
    output[cols] = h00 * y1 + h10 * h * m1 + h01 * y2 + h11 * h * m2
    method[cols] = 1
    return output, method

### **3. Verifikasi PCHIP manual vs SciPy**

In [ ]:
# ============================================================
# Uji 1: semua anchor valid (1 piksel) -> bandingkan dengan SciPy
# Hanya untuk pengujian, bukan proses utama
# ============================================================
from scipy.interpolate import PchipInterpolator

rng = np.random.default_rng(42)
max_error, total_test = 0.0, 0
for n_anchor in [2, 3, 4, 6, 8]:
    for _ in range(100):
        x = np.cumsum(rng.uniform(5, 100, n_anchor))
        y = rng.uniform(500, 10000, n_anchor)
        target = rng.uniform(x[0] + 0.001, x[-1] - 0.001)
        manual, _method = pchip_manual_from_anchors(x, y[:, None], np.ones((n_anchor, 1), dtype=bool), target)
        scipy_result = PchipInterpolator(x, y, extrapolate=False)(target)
        max_error = max(max_error, float(abs(manual[0] - scipy_result)))
        total_test += 1
        if not np.allclose(manual[0], scipy_result, rtol=1e-10, atol=1e-8):
            raise AssertionError("PCHIP manual berbeda dengan SciPy.")

print(f"[OK] {total_test} pengujian berhasil.")
print(f"Maximum numerical error: {max_error:.12f}")
print("PCHIP manual sesuai dengan SciPy pada kasus pengujian.")

In [ ]:
# ============================================================
# Uji 2: banyak piksel sekaligus, sebagian anchor TIDAK valid per piksel
# (meniru kondisi nyata: anchor berawan di sebagian piksel)
# Acuan per piksel: SciPy PCHIP pada anchor valid saja; backward/forward fill
# = nilai anchor valid terdekat jika target di luar rentang anchor valid.
# ============================================================
rng = np.random.default_rng(7)
n_anchor, n_pixel, n_trial = 10, 2000, 20
max_error, method_count = 0.0, np.zeros(4, dtype=int)

for _ in range(n_trial):
    x = np.sort(rng.choice(np.arange(0, 3650), n_anchor, replace=False)).astype(np.float64)
    y = rng.uniform(500, 10000, (n_anchor, n_pixel))
    valid = rng.random((n_anchor, n_pixel)) > 0.4
    target = float(rng.uniform(x[0] - 50, x[-1] + 50)) + 0.5  # +0.5 agar tidak sama dengan anchor
    manual, method = pchip_manual_from_anchors(x, y, valid, target)

    for p in range(n_pixel):
        xv, yv = x[valid[:, p]], y[valid[:, p], p]
        before, after = xv < target, xv > target
        if not before.any() and not after.any():
            exp_val, exp_method = np.nan, 0
        elif not before.any():
            exp_val, exp_method = yv[after][0], 2
        elif not after.any():
            exp_val, exp_method = yv[before][-1], 3
        else:
            exp_val, exp_method = PchipInterpolator(xv, yv)(target), 1
        if method[p] != exp_method:
            raise AssertionError(f"Metode berbeda pada piksel {p}: {method[p]} vs {exp_method}")
        if exp_method != 0:
            if not np.isclose(manual[p], exp_val, rtol=1e-10, atol=1e-8):
                raise AssertionError(f"Nilai berbeda pada piksel {p}: {manual[p]} vs {exp_val}")
            max_error = max(max_error, abs(manual[p] - exp_val))
        method_count[exp_method] += 1

print(f"[OK] {n_trial * n_pixel:,} piksel diuji.")
print(f"Jumlah per metode -> tidak terisi: {method_count[0]:,} | PCHIP: {method_count[1]:,} | "
      f"backward: {method_count[2]:,} | forward: {method_count[3]:,}")
print(f"Maximum numerical error: {max_error:.12f}")

### **4. Persiapan raster dan pemrosesan blockwise**

In [ ]:
# ============================================================
# Index file raster & validasi grid
# ============================================================
def build_region_file_index(region_dir):
    """Petakan setiap tanggal -> path raster 9 band + SCL."""
    dates = get_region_dates(region_dir)
    files_by_date = {}
    for date_name in dates:
        folder = os.path.join(region_dir, date_name)
        files_by_date[date_name] = {
            "bands": {b: find_one_file(os.path.join(folder, f"*_{b}_10m.tif"), f"{date_name} {b}")
                      for b in BANDS_TO_PROCESS},
            "scl": find_one_file(os.path.join(folder, "*_SCL_10m.tif"), f"{date_name} SCL"),
        }
    return dates, files_by_date


def validate_region_grid(region_name, dates, files_by_date):
    """Pastikan semua raster (band + SCL, semua tanggal) punya ukuran, CRS, dan transform sama."""
    ref_path = files_by_date[dates[0]]["bands"]["B04"]
    with rasterio.open(ref_path) as ref:
        ref_width, ref_height, ref_crs, ref_transform = ref.width, ref.height, ref.crs, ref.transform
    print(f"\nValidasi grid {region_name}")
    print(f"Referensi : {os.path.basename(ref_path)}")
    print(f"Ukuran    : {ref_width} x {ref_height}")
    print(f"CRS       : {ref_crs}")
    print(f"Resolusi  : ({ref_transform.a}, {abs(ref_transform.e)})")

    total_checked = 0
    for date_name in dates:
        paths = [files_by_date[date_name]["bands"][b] for b in BANDS_TO_PROCESS]
        paths.append(files_by_date[date_name]["scl"])
        for path in paths:
            with rasterio.open(path) as src:
                same_grid = (src.width == ref_width and src.height == ref_height
                             and src.crs == ref_crs and src.transform.almost_equals(ref_transform))
                if not same_grid:
                    raise RuntimeError(f"\nGrid raster tidak sama.\nFile: {path}\n"
                                       f"Size: {src.width}x{src.height}\nCRS: {src.crs}\n"
                                       f"Transform: {src.transform}")
            total_checked += 1
    print(f"[OK] {total_checked} raster berada pada grid yang sama.")

# ============================================================
# Salin SCL asli & marker proses
# ============================================================
def get_scl_output_path(region_out_dir, date_name, scl_source_path):
    filename = os.path.basename(scl_source_path)
    if filename.endswith("_SCL_10m.tif"):
        filename = filename.replace("_SCL_10m.tif", "_SCL_asli_10m.tif")
    else:
        filename = filename.replace(".tif", "_SCL_asli_10m.tif")
    return os.path.join(region_out_dir, date_name, filename)


def prepare_scl_outputs(region_out_dir, dates, files_by_date):
    print("\nMenyalin SCL asli...")
    for date_name in dates:
        os.makedirs(os.path.join(region_out_dir, date_name), exist_ok=True)
        scl_source = files_by_date[date_name]["scl"]
        scl_output = get_scl_output_path(region_out_dir, date_name, scl_source)
        if FORCE_REPROCESS or not os.path.exists(scl_output):
            shutil.copy2(scl_source, scl_output)
    print("[OK] SCL asli selesai disalin.")


def band_marker_path(region_out_dir, band_code):
    return os.path.join(region_out_dir, f".DONE_{band_code}")


def marker_is_current(marker_path):
    """True jika band sudah selesai diproses dengan PROCESS_VERSION yang sama."""
    if not os.path.exists(marker_path):
        return False
    try:
        with open(marker_path, "r") as f:
            return f.read().strip() == PROCESS_VERSION
    except Exception:
        return False

# ============================================================
# Siapkan output & casting nilai
# ============================================================
def initialize_band_outputs(region_out_dir, dates, files_by_date, band_code, reset_band=False):
    """Salin raster band asli ke folder output; piksel berawan nanti ditimpa hasil gap filling."""
    out_paths = {}
    for date_name in dates:
        source_path = files_by_date[date_name]["bands"][band_code]
        out_folder = os.path.join(region_out_dir, date_name)
        os.makedirs(out_folder, exist_ok=True)
        out_path = os.path.join(out_folder, os.path.basename(source_path))
        if reset_band or not os.path.exists(out_path):  # copy ulang dari raster asli
            shutil.copy2(source_path, out_path)
        out_paths[date_name] = out_path
    return out_paths


def cast_values_to_dtype(values, dtype):
    """Bulatkan & clip hasil float ke dtype raster asli (mis. uint16)."""
    dtype = np.dtype(dtype)
    if np.issubdtype(dtype, np.integer):
        info = np.iinfo(dtype)
        return np.clip(np.rint(values), info.min, info.max).astype(dtype)
    return values.astype(dtype)

# ============================================================
# Proses 1 band secara blockwise
# ============================================================
def process_band_blockwise(region_name, region_out_dir, dates, files_by_date, band_code, anchor_dates):
    print_header(f"PROCESS BAND {band_code} - {region_name}")
    marker_path = band_marker_path(region_out_dir, band_code)
    band_log_path = os.path.join(region_out_dir, f"log_{band_code}.csv")

    # Skip jika band sudah selesai dengan versi proses yang sama
    if marker_is_current(marker_path) and not FORCE_REPROCESS:
        print(f"[skip] {band_code} sudah selesai dengan versi {PROCESS_VERSION}.")
        return pd.read_csv(band_log_path) if os.path.exists(band_log_path) else pd.DataFrame()

    # Mulai ulang band dari raster ASLI
    out_paths = initialize_band_outputs(region_out_dir, dates, files_by_date, band_code, reset_band=True)
    anchor_dates = sorted(anchor_dates)
    anchor_positions = [dates.index(d) for d in anchor_dates]
    anchor_times = np.array([date_to_ordinal(d) for d in anchor_dates], dtype=np.float64)
    target_times = {d: date_to_ordinal(d) for d in dates}

    print(f"Total tanggal : {len(dates)}")
    print(f"Total anchor  : {len(anchor_dates)}")
    print(f"Cloud buffer  : {CLOUD_BUFFER_METERS:.0f} meter")
    print(f"Block size    : {BLOCK_SIZE} x {BLOCK_SIZE}")

    stat_keys = ["final_cloud_pixels", "pchip_pixels", "backward_fill_pixels",
                 "forward_fill_pixels", "unfilled_pixels"]
    stats = {d: dict.fromkeys(stat_keys, 0) for d in dates}
    start_time = time.time()

    with ExitStack() as stack:
        def open_raster(path, mode="r"):
            return stack.enter_context(rasterio.open(path, mode))

        scl_srcs = {d: open_raster(files_by_date[d]["scl"]) for d in dates}
        anchor_srcs = {d: open_raster(files_by_date[d]["bands"][band_code]) for d in anchor_dates}  # raster ASLI
        out_dsts = {d: open_raster(out_paths[d], "r+") for d in dates}

        width, height = out_dsts[dates[0]].width, out_dsts[dates[0]].height
        total_windows = int(np.ceil(width / BLOCK_SIZE)) * int(np.ceil(height / BLOCK_SIZE))
        print(f"Total window : {total_windows}")

        for window_number, window in enumerate(iter_windows(width, height), start=1):
            # 1) Mask awan final (SCL + buffer 30 m) seluruh tanggal
            final_cloud_list, valid_list = [], []
            for d in dates:
                final_cloud, valid, _, _ = read_final_cloud_mask_window(scl_srcs[d], window)
                final_cloud_list.append(final_cloud)
                valid_list.append(valid)
            final_cloud_stack = np.stack(final_cloud_list, axis=0)
            valid_stack = np.stack(valid_list, axis=0)

            # 2) Nilai anchor ASLI + validitas per piksel
            anchor_values = np.stack([anchor_srcs[d].read(1, window=window).astype(np.float32, copy=False)
                                      for d in anchor_dates], axis=0)
            anchor_valid = (valid_stack[anchor_positions] & ~final_cloud_stack[anchor_positions]
                            & np.isfinite(anchor_values))
            for i, d in enumerate(anchor_dates):  # keluarkan piksel NoData anchor dari donor
                nodata = anchor_srcs[d].nodata
                if nodata is None:
                    continue
                if isinstance(nodata, float) and np.isnan(nodata):
                    anchor_valid[i] &= np.isfinite(anchor_values[i])
                else:
                    anchor_valid[i] &= anchor_values[i] != nodata
            anchor_flat = anchor_values.reshape(len(anchor_dates), -1)
            valid_flat = anchor_valid.reshape(len(anchor_dates), -1)

            # 3) Isi piksel berawan pada setiap tanggal target
            for date_idx, date_name in enumerate(dates):
                target_cloud = final_cloud_stack[date_idx]
                if not np.any(target_cloud):
                    continue
                dst = out_dsts[date_name]
                out_block = dst.read(1, window=window)
                out_flat = out_block.reshape(-1)  # view: perubahan out_flat ikut ke out_block
                cloud_pixels = np.flatnonzero(target_cloud.reshape(-1))

                filled_values, method = pchip_manual_from_anchors(
                    anchor_times=anchor_times,
                    anchor_values=anchor_flat[:, cloud_pixels],
                    anchor_valid=valid_flat[:, cloud_pixels],
                    target_time=target_times[date_name],
                )
                filled = np.isfinite(filled_values)
                if np.any(filled):
                    out_flat[cloud_pixels[filled]] = cast_values_to_dtype(filled_values[filled], dst.dtypes[0])
                if np.any(~filled):  # tidak bisa diisi -> NoData (atau 0)
                    nodata_value = dst.nodata if dst.nodata is not None else 0
                    out_flat[cloud_pixels[~filled]] = np.asarray(nodata_value, dtype=dst.dtypes[0])
                dst.write(out_block, 1, window=window)

                s = stats[date_name]
                s["final_cloud_pixels"] += int(len(cloud_pixels))
                s["pchip_pixels"] += int(np.sum(method == 1))
                s["backward_fill_pixels"] += int(np.sum(method == 2))
                s["forward_fill_pixels"] += int(np.sum(method == 3))
                s["unfilled_pixels"] += int(np.sum(method == 0))

            if window_number == 1 or window_number % 10 == 0 or window_number == total_windows:
                print(f"[{band_code}] window {window_number}/{total_windows} "
                      f"| elapsed {(time.time() - start_time) / 60:.1f} menit")

            del final_cloud_stack, valid_stack, anchor_values, anchor_valid, anchor_flat, valid_flat

    # Simpan log & marker
    rows = [{"region": region_name, "band": band_code, "tanggal": d, "process_version": PROCESS_VERSION,
             "cloud_buffer_m": CLOUD_BUFFER_METERS, **stats[d]} for d in dates]
    df_log = pd.DataFrame(rows)
    df_log.to_csv(band_log_path, index=False)
    with open(marker_path, "w") as f:
        f.write(PROCESS_VERSION)

    elapsed = time.time() - start_time
    print(f"\n[OK] {band_code} selesai.")
    print(f"Final cloud/buffer : {int(df_log['final_cloud_pixels'].sum()):,}")
    print(f"PCHIP              : {int(df_log['pchip_pixels'].sum()):,}")
    print(f"Backward fill      : {int(df_log['backward_fill_pixels'].sum()):,}")
    print(f"Forward fill       : {int(df_log['forward_fill_pixels'].sum()):,}")
    print(f"Tidak dapat diisi  : {int(df_log['unfilled_pixels'].sum()):,}")
    print(f"Waktu              : {elapsed / 60:.2f} menit")
    gc.collect()
    return df_log

### **5. Main process**

In [ ]:
all_logs = []
for region_name, region_dir in REGION_DIRS.items():
    region_out_dir = PCHIP_OUT_DIRS[region_name]
    os.makedirs(region_out_dir, exist_ok=True)
    print_header(f"# WILAYAH: {region_name}", "#")
    print(f"Input        : {region_dir}")
    print(f"Output       : {region_out_dir}")
    print(f"Cloud buffer : {CLOUD_BUFFER_METERS:.0f} meter")

    dates, files_by_date = build_region_file_index(region_dir)
    print(f"Total tanggal ditemukan: {len(dates)}")

    anchors = [d for d in anchor_dates_per_region[region_name] if d in dates]
    if len(anchors) < 2:
        raise RuntimeError(f"{region_name} hanya punya {len(anchors)} anchor valid setelah buffer.")
    print(f"Total anchor digunakan: {len(anchors)}")
    print(anchors)

    validate_region_grid(region_name, dates, files_by_date)
    prepare_scl_outputs(region_out_dir, dates, files_by_date)

    for band_code in BANDS_TO_PROCESS:  # proses 9 band
        df_band_log = process_band_blockwise(region_name, region_out_dir, dates, files_by_date,
                                             band_code, anchors)
        if df_band_log is not None and len(df_band_log) > 0:
            all_logs.append(df_band_log)
        gc.collect()

# Gabungkan log seluruh wilayah & band
if all_logs:
    df_all_logs = pd.concat(all_logs, ignore_index=True)
    log_path = os.path.join(SKRIPSI_DIR, "pchip_processing_log_buffer30m.csv")
    df_all_logs.to_csv(log_path, index=False)
    print_header("SELURUH PROSES SELESAI")
    print(f"Log: {log_path}")
else:
    print("\nSemua band sebelumnya sudah selesai.")